In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from concurrent.futures import ProcessPoolExecutor
from scipy.integrate import solve_ivp


def system(t, u):
    x, y, p, q = u
    
    dxdt = p
    dydt = q
    dpdt = y**2 - x**2 - x
    dqdt = 2*x*y - y
    
    return [dxdt, dydt, dpdt, dqdt]


def escape(t, u, R=1.5):
    x, y, p, q = u
    return R**2 - (x**2 + y**2)
escape.terminal=True


    
def solve_clover(u0, t_max=300, t_steps=10000):
    t_eval = np.linspace(0, t_max, t_steps)
    
    sol = solve_ivp(
        system,                
        [0, t_max],      
        u0,                     
        t_eval=t_eval,          
        rtol=1e-12, atol=1e-12,
        events=[escape],
        max_step=0.1
    )
    
    return sol


def categorise_trajectory(u0: list[float]) -> int:
    """
    Categorises a trajectory into one of four categories:
    - 0: Does not diverge
    - 1: Diverges towards the left
    - 2: Diverges towards the top right
    - 3: Diverges towards the bottom right
    """
    # Assuming solve_clover is defined elsewhere in your script
    solution = solve_clover(u0)

    if solution.status == 0:
        return 0
    
    escape_state = solution.y_events[0][0] 
    
    x_escape = escape_state[0]
    y_escape = escape_state[1]

    if x_escape < 0:
        return 1
    elif y_escape > 0:
        return 2 
    else:
        return 3

def create_exit_basin_plot(n=100, max_workers=None):
    xs = np.linspace(-1, 0.5, n)
    qs = np.linspace(0.5, 1, n)
    
    X, Q = np.meshgrid(xs, qs)

    # 1. Flatten the inputs into a list of tasks
    # We create a 1D list of every u0 combination we need to solve
    u0_list = [[X[i, j], 0.0, 0.0, Q[i, j]] for i in range(n) for j in range(n)]

    print(f"Starting parallel evaluation of {n*n} trajectories...")

    # 2. Distribute the tasks across multiple CPU cores
    # ProcessPoolExecutor bypasses Python's Global Interpreter Lock (GIL)
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # executor.map automatically applies the function to the list 
        # and guarantees the results are returned in the exact original order.
        results = list(executor.map(categorise_trajectory, u0_list))

    print("Evaluations complete. Rendering plot...")

    # 3. Reshape the flat 1D list of results back into the 2D grid
    C = np.array(results).reshape((n, n))

    # --- Plotting remains identical ---
    cmap = ListedColormap([
        "black",   
        "#82B2E6",
        "#BF62E3", 
        "#F58518"
    ])
    labels = [
        "0: bounded",
        "1: left divergence",
        "2: top right divergence",
        "3: bottom right divergence"
    ]

    patches = [
        mpatches.Patch(color=cmap(i), label=labels[i])
        for i in range(4)
    ]

    plt.figure(figsize=(8, 6)) # Added a standard figsize for better rendering
    plt.pcolormesh(X, Q, C, cmap=cmap, shading='auto')

    plt.xlabel("x")
    plt.ylabel("q")
    plt.title("Exit Basin Plot")
    plt.legend(handles=patches, loc="upper right", prop={'size': 7})

    plt.show()


# CRITICAL: When using multiprocessing in Python, you MUST protect the entry point
# of your script with this if-statement, otherwise it will crash on Windows/macOS.
if __name__ == '__main__':
    # You can increase 'n' now that it's parallelized!
    # max_workers=None will automatically use all available CPU cores.
    create_exit_basin_plot(n=100)

Starting parallel evaluation of 10000 trajectories...


BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.